<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# DATA/MSML 641: Natural Language Processing
## Lecture 8: Transformers

**University of Maryland, College Park**  
**Spring 2026**  
**Instructor**: Armin Mehrabian  
**Date**: April 7-8, 2026  

<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# Part 1: Introduction to Transformers

In this lecture, we will learn about **Transformers**, the architecture that powers modern Large Language Models (LLMs).

# LLMs are built out of transformers

**Transformer**: a specific kind of network architecture, like a fancier feedforward network, but based on attention.

- The foundational paper that introduced transformers
- Relies on self-attention


<div style="text-align: left; margin-bottom: 20px;">
  <img src="img/attention_is_all_you_need.png" 
       alt="Attention" style="height: auto;" />
</div>

# Part 1: Attention

Before diving into the full transformer architecture, let's understand the key mechanism that makes it work: **attention**.

# Problem with static embeddings (word2vec)

They are **static**! The embedding for a word doesn't reflect how its meaning changes in context.

Example:

> *The chicken didn't cross the road because **it** was too tired*

What is the meaning represented in the static embedding for "it"?

The static embedding cannot capture whether "it" refers to the chicken or the road - it's always the same vector regardless of context.

# Contextual Embeddings

**Key Intuition**: A representation of the meaning of a word should be different in different contexts!

**Contextual Embedding**: Each word has a different vector that expresses different meanings depending on the surrounding words.

**How to compute contextual embeddings?**
- **Attention**

# Contextual Embeddings: Example

Consider:

> *The chicken didn't cross the road because **it***

**What should be the properties of "it"?**

At this point in the sentence, it's probably referring to either the chicken or the road.

Compare:
- *The chicken didn't cross the road because it was too **tired*** → "it" refers to the chicken
- *The chicken didn't cross the road because it was too **wide*** → "it" refers to the road

# Intuition of attention

Build up the contextual embedding for a word by **selectively integrating information** from all the neighboring words.

We say that a word **"attends to"** some neighboring words more than others.

# Intuition of attention

<center>
<img width=60% src="img/attention_distrib.png">
</center>

```
- Layer k with tokens: "The", "chicken", "didn't", "cross", "the", "road", "because", "it", "was", "too", "tired"
- Layer k+1 with same tokens
- Arrows showing "it" attending primarily to "chicken" and "road" with a "self-attention distribution"]
```

The word "it" in layer k+1 is formed by attending to relevant words from layer k.

# Intuition of attention

<center>
<img width=80% src="img/self_attention_layer.png">
</center>

# Simplified version of attention

A sum of prior words weighted by their similarity with the current word.

Given a sequence of token embeddings:
$$x_1 \quad x_2 \quad x_3 \quad x_4 \quad x_5 \quad x_6 \quad x_7 \quad x_i$$

Produce: $\mathbf{a}_i$ = a weighted sum of $x_1$ through $x_7$ (and $x_i$)

Weighted by their similarity to $x_i$

$$\text{score}(x_i, x_j) = x_i \cdot x_j$$

$$\alpha_{ij} = \text{softmax}(\text{score}(x_i, x_j)) \quad \forall j \leq i$$

$$\mathbf{a}_i = \sum_{j \leq i} \alpha_{ij} x_j$$

# An Actual Attention Head: slightly more complicated

High-level idea: instead of using vectors (like $x_i$ and $x_4$) directly, we'll represent 3 separate roles each vector $x_i$ plays:

- **query**: As *the current element* being compared to the preceding inputs
- **key**: As *a preceding input* that is being compared to the current element to determine a similarity
- **value**: A value of a preceding element that gets weighted and summed

<center>
<img width=80% src="img/attention_qv.png">
</center>

<center>
<img width=80% src="img/attention_q_k_v.png">
</center>

# An Actual Attention Head: Projection Matrices

We'll use matrices to project each vector $x_i$ into a representation of its role as query, key, value:

- **query**: $\mathbf{W}^Q$
- **key**: $\mathbf{W}^K$
- **value**: $\mathbf{W}^V$

$$\mathbf{q}_i = x_i \mathbf{W}^Q; \quad \mathbf{k}_i = x_i \mathbf{W}^K; \quad \mathbf{v}_i = x_i \mathbf{W}^V$$

# One attention head

$$\mathbf{q}_i = x_i \mathbf{W}^Q; \quad \mathbf{k}_j = x_j \mathbf{W}^K; \quad \mathbf{v}_j = x_j \mathbf{W}^V$$

$$\text{score}(x_i, x_j) = \frac{\mathbf{q}_i \cdot \mathbf{k}_j}{\sqrt{d_k}}$$

$$\alpha_{ij} = \text{softmax}(\text{score}(x_i, x_j)) \quad \forall j \leq i$$

$$\mathbf{a}_i = \sum_{j \leq i} \alpha_{ij} \mathbf{v}_j$$

In [4]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
# Simple attention mechanism demonstration
 
    return output, weights

# Example usage
d = 4  # dimension
query = np.random.randn(1, d)
keys = np.random.randn(3, d)
values = np.random.randn(3, d)

output, weights = simple_attention(query, keys, values)

print("Attention weights:", weights)
print("Output shape:", output.shape)

Attention weights: [[0.56197505 0.22087953 0.21714542]]
Output shape: (1, 4)


# Example: Calculating the value of a3

<center>
<img width=80% src="img/a3_self_attention.png">
</center>

1. Generate key, query, value vectors from x1, x2, x3 using WK, WQ, WV
2. Compare x3's query with the keys for x1, x2, and x3 (dot products)
3. Divide scores by √dk
4. Turn into weights via softmax (α3,1, α3,2, α3,3)
5. Weigh each value vector
6. Sum the weighted value vectors to get a3]


# Multi-head attention

Instead of one attention head, we'll have **lots of them**!

**Intuition**: Each head might be attending to the context for different purposes
- Different linguistic relationships or patterns in the context

$$\mathbf{q}_i^c = x_i \mathbf{W}^{Q_c}; \quad \mathbf{k}_j^c = x_j \mathbf{W}^{K_c}; \quad \mathbf{v}_j^c = x_j \mathbf{W}^{V_c}; \quad \forall c \quad 1 \leq c \leq h$$

$$\text{score}^c(x_i, x_j) = \frac{\mathbf{q}_i^c \cdot \mathbf{k}_j^c}{\sqrt{d_k}}$$

$$\alpha_{ij}^c = \text{softmax}(\text{score}^c(x_i, x_j)) \quad \forall j \leq i$$

$$\text{head}_i^c = \sum_{j \leq i} \alpha_{ij}^c \mathbf{v}_j^c$$

$$\mathbf{a}_i = (\text{head}_i^1 \oplus \text{head}_i^2 ... \oplus \text{head}_i^h) \mathbf{W}^O$$

$$\text{MultiHeadAttention}(x_i, [x_1, \cdots, x_N]) = \mathbf{a}_i$$

# Multi-head attention

<center>
<img width=80% src="img/multi_headed_attention.png">
</center>

# Summary of Attention

Attention is a method for enriching the representation of a token by incorporating contextual information.

**The result**: The embedding for each word will be different in different contexts!

**Contextual embeddings**: A representation of word meaning in its context.

We'll see in the next section that attention can also be viewed as a way to move information from one token to another.

<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# Part 2: The Transformer Architecture

Now let's see how attention is integrated into transformer blocks that form the building blocks of modern LLMs.

<center>
<img width=80% src="img/transformer-block.png">
</center>

# Transformer language model


- Input tokens: "So", "long", "and", "thanks", "for"
- Input Encoding: E + positional embeddings (1, 2, 3, 4, 5)
- Stacked Transformer Blocks (multiple layers)
- Language Modeling Head outputting logits for next token prediction
- Next token predictions: "long", "and", "thanks", "for", "all"]


<center>
<img width=80% src="img/transformer_language_model.png">
</center>

# Feedforward Layer (FFN)

Transformers also need **nonlinearities** — this is where the feedforward layer comes in.

$$\text{FFN}(x_i) = \text{ReLU}(x_i \mathbf{W}_1 + b_1)\mathbf{W}_2 + b_2$$

> **Note**: Modern transformers (Llama, Mistral, etc.) typically replace ReLU with **SwiGLU** or **GELU** activations, which empirically improve performance. The SwiGLU variant uses a gated linear unit:
> $$\text{FFN}_{\text{SwiGLU}}(x) = (x\mathbf{W}_1 \odot \text{SiLU}(x\mathbf{W}_{\text{gate}}))\mathbf{W}_2$$

# The residual stream: each token gets passed up and modified

<center>
<img width=80% src="img/residual_stream.png">
</center>


- Input xi at bottom
- Flows up through hi-1, hi, hi+1
- Two modification points marked with ⊕:
  1. After Layer Norm → MultiHead Attention
  2. After Layer Norm → Feedforward
- Both modifications are added back to the residual stream]


Each token has its own **residual stream** that flows upward through the network, getting modified at each layer.

# The vector xi is normalized twice

<center>
<img width=80% src="img/residual_stream.png">
</center>


Same residual stream diagram highlighting both Layer Norm components (before attention and before feedforward)

# Layer Norm

Layer norm is a variation of the z-score from statistics, applied to a single vector in a hidden layer.

$$\mu = \frac{1}{d} \sum_{i=1}^{d} x_i$$

$$\sigma = \sqrt{\frac{1}{d} \sum_{i=1}^{d} (x_i - \mu)^2}$$

$$\hat{x} = \frac{(x - \mu)}{\sigma}$$

$$\text{LayerNorm}(x) = \gamma \frac{(x - \mu)}{\sigma} + \beta$$

where $\gamma$ and $\beta$ are learnable parameters.

In [12]:
# Layer Normalization implementation
def layer_norm(x, gamma, beta, eps=1e-5):
    """
    Apply layer normalization.
    
    Args:
        x: Input vector
        gamma: Scale parameter
        beta: Shift parameter
        eps: Small constant for numerical stability
    
    Returns:
        Normalized vector
    """
    mean = np.mean(x)
    var = np.var(x)
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta

# Example
x = np.random.randn(10)
gamma = np.ones(10)
beta = np.zeros(10)

normalized = layer_norm(x, gamma, beta)
print(f"Original mean: {np.mean(x):.4f}, std: {np.std(x):.4f}")
print(f"Normalized mean: {np.mean(normalized):.4f}, std: {np.std(normalized):.4f}")

Original mean: -0.1354, std: 0.8971
Normalized mean: 0.0000, std: 1.0000


# Putting together a single transformer block


$$\mathbf{t}_i^1 = \text{LayerNorm}(x_i)$$

$$\mathbf{t}_i^2 = \text{MultiHeadAttention}(\mathbf{t}_i^1, [x_1^1, \cdots, x_N^1])$$

$$\mathbf{t}_i^3 = \mathbf{t}_i^2 + x_i$$

$$\mathbf{t}_i^4 = \text{LayerNorm}(\mathbf{t}_i^3)$$

$$\mathbf{t}_i^5 = \text{FFN}(\mathbf{t}_i^4)$$

$$\mathbf{h}_i = \mathbf{t}_i^5 + \mathbf{t}_i^3$$

<center>
<img width=80% src="img/residual_stream.png">
</center>

# A transformer is a stack of these blocks

All the vectors are of the same dimensionality $d$


<center>
<img width=80% src="img/2_attention_blocks.png">
</center>

Two transformer blocks stacked vertically (Block 1 and Block 2), showing the same internal structure for each block


# Residual streams and attention

Notice that all parts of the transformer block apply to **1 residual stream** (1 token).

**Except attention**, which takes information from other tokens.

Elhage et al. (2021) show that we can view attention heads as literally **moving information** from the residual stream of a neighboring token into the current stream.

<center>
<img width=80% src="img/residual_streams_attention.png">
</center>

- Multiple vertical bars representing residual streams for different tokens
- Arrow showing information flowing from Token A residual stream to Token B residual stream via attention]


<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# Part 3: Parallelizing Attention Computation

One of the key advantages of transformers is the ability to parallelize computation efficiently.

# Parallelizing computation using X

For attention/transformer block we've been computing a **single** output at a **single** time step $i$ in a **single** residual stream.

But we can pack the $N$ tokens of the input sequence into a single matrix $\mathbf{X}$ of shape $[N \times d]$.

- Each row of $\mathbf{X}$ is the embedding of one token of the input
- $\mathbf{X}$ can have 1K-32K rows, each of the dimensionality of the embedding $d$ (the model dimension)

$$\mathbf{Q} = \mathbf{X}\mathbf{W}^Q; \quad \mathbf{K} = \mathbf{X}\mathbf{W}^K; \quad \mathbf{V} = \mathbf{X}\mathbf{W}^V$$

# QKᵀ

Now we can do a single matrix multiply to combine $\mathbf{Q}$ and $\mathbf{K}^T$


<center>
<img width=80% src="img/QK-matrix.png">
</center>

- Rows labeled q1, q2, q3, q4
- Columns labeled k1, k2, k3, k4
- Each cell contains the dot product q·k (e.g., q1·k1, q1·k2, etc.)]


This $N \times N$ matrix contains all pairwise query-key similarities.

# Parallelizing attention

Scale the scores, take the softmax, and then multiply the result by $\mathbf{V}$ resulting in a matrix of shape $N \times d$

- An attention vector for each input token

$$\mathbf{A} = \text{softmax}\left(\text{mask}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\right) \mathbf{V}$$

# Masking out the future

$$\mathbf{A} = \text{softmax}\left(\text{mask}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\right) \mathbf{V}$$

**What is this mask function?**

$\mathbf{Q}\mathbf{K}^T$ has a score for each query dot every key, *including those that follow the query*.

Guessing the next word is pretty simple if you already know it!

# Masking out the future

Add $-\infty$ to cells in upper triangle. The softmax will turn it to 0.


- Lower triangle filled with actual values (q1·k1, q2·k1, q2·k2, etc.)
- Upper triangle filled with -∞

<center>
<img width=80% src="img/masked-QK-matrix.png">
</center>

In [17]:
# Causal mask demonstration
def create_causal_mask(size):
    """
    Create a causal mask for attention.
    
    Args:
        size: Size of the mask (number of tokens)
    
    Returns:
        Mask matrix where upper triangle is -inf
    """
    mask = np.triu(np.ones((size, size)) * -np.inf, k=1)
    return mask

# Example
mask = create_causal_mask(5)
print("Causal mask:")
print(mask)

# Apply to scores
scores = np.random.randn(5, 5)
masked_scores = scores + mask

print("\nScores after masking:")
print(masked_scores)

Causal mask:
[[  0. -inf -inf -inf -inf]
 [  0.   0. -inf -inf -inf]
 [  0.   0.   0. -inf -inf]
 [  0.   0.   0.   0. -inf]
 [  0.   0.   0.   0.   0.]]

Scores after masking:
[[ 2.65926827e+00            -inf            -inf            -inf
             -inf]
 [ 3.55443709e-01 -1.15407436e-03            -inf            -inf
             -inf]
 [ 3.09842048e-01  1.24571544e+00  3.21458498e-01            -inf
             -inf]
 [-8.78817096e-01  1.33980578e+00 -2.60448448e-01 -8.28352705e-01
             -inf]
 [ 7.34803848e-01  1.74363095e-01  9.69820447e-01 -1.64282586e+00
   2.43862626e-01]]


# Another point: Attention is quadratic in length

$$\mathbf{A} = \text{softmax}\left(\text{mask}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\right) \mathbf{V}$$

The $\mathbf{Q}\mathbf{K}^T$ matrix has shape $N \times N$, where $N$ is the sequence length.

This means:
- Memory requirements scale as $O(N^2)$
- Computation scales as $O(N^2)$

This is why context length is a major bottleneck for transformers!

**How modern systems address this:**
- **Flash Attention** (Dao et al., 2022): Restructures computation to minimize GPU memory reads/writes — computes *exact* attention 2-4x faster without ever materializing the full N×N matrix in memory
- **Sliding Window Attention**: Each token only attends to a local window (e.g., 4096 tokens); information propagates through stacked layers
- **Grouped-Query Attention (GQA)**: Shares K/V heads across query heads, reducing the KV cache memory bottleneck at inference
- **Linear attention / SSMs**: Replace softmax attention with O(N) alternatives (e.g., Mamba)

# Attention again: complete picture

<center>
<img width=80% src="img/masked-full-attention.png">
</center>

- Input X (N×d) split into three paths
- Multiply by WQ (d×dk), WK (d×dk), WV (d×dv) to get Q, K, V
- Q (N×dk) × Kᵀ (dk×N) = QKᵀ (N×N)
- Apply mask to upper triangle
- Softmax to get attention weights
- Multiply by V (N×dv) to get A (N×dv)]


# Parallelizing Multi-head Attention

$$\mathbf{Q}^i = \mathbf{X}\mathbf{W}^{Q_i}; \quad \mathbf{K}^i = \mathbf{X}\mathbf{W}^{K_i}; \quad \mathbf{V}^i = \mathbf{X}\mathbf{W}^{V_i}$$

$$\text{head}_i = \text{SelfAttention}(\mathbf{Q}^i, \mathbf{K}^i, \mathbf{V}^i) = \text{softmax}\left(\frac{\mathbf{Q}^i {\mathbf{K}^i}^T}{\sqrt{d_k}}\right) \mathbf{V}^i$$

$$\text{MultiHeadAttention}(\mathbf{X}) = (\text{head}_1 \oplus \text{head}_2 ... \oplus \text{head}_h)\mathbf{W}^O$$

# Parallelizing Multi-head Attention: Complete Block

> **Note on normalization order**: The equations below show **post-norm** (LayerNorm after residual addition), which was used in the original "Attention Is All You Need" paper. Most modern transformers use **pre-norm** (LayerNorm *before* attention/FFN, as shown in the single-block slide earlier), which improves training stability.

Matrix form of one transformer block (post-norm variant):

$$\mathbf{T}^1 = \text{MultiHeadAttention}(\mathbf{X})$$

$$\mathbf{T}^2 = \mathbf{X} + \mathbf{T}^1$$

$$\mathbf{T}^3 = \text{LayerNorm}(\mathbf{T}^2)$$

$$\mathbf{T}^4 = \text{FFN}(\mathbf{T}^3)$$

$$\mathbf{T}^5 = \mathbf{T}^4 + \mathbf{T}^3$$

$$\mathbf{H} = \text{LayerNorm}(\mathbf{T}^5)$$

In [18]:
# Simplified multi-head attention implementation
def multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads, d_k):
    """
    Multi-head attention mechanism.
    
    Args:
        X: Input matrix (N x d)
        W_Q, W_K, W_V: Weight matrices for queries, keys, values
        W_O: Output projection matrix
        num_heads: Number of attention heads
        d_k: Dimension of each head
    
    Returns:
        Output after multi-head attention
    """
    N, d = X.shape
    
    # Compute Q, K, V for all heads
    Q = np.dot(X, W_Q)
    K = np.dot(X, W_K)
    V = np.dot(X, W_V)
    
    # Reshape for multiple heads
    Q = Q.reshape(N, num_heads, d_k)
    K = K.reshape(N, num_heads, d_k)
    V = V.reshape(N, num_heads, d_k)
    
    # Compute attention for each head
    heads = []
    for h in range(num_heads):
        # Attention scores
        scores = np.dot(Q[:, h, :], K[:, h, :].T) / np.sqrt(d_k)
        
        # Apply causal mask
        mask = create_causal_mask(N)
        scores = scores + mask
        
        # Softmax (numerically stable)
        scores_shifted = scores - np.max(scores, axis=-1, keepdims=True)
        attention_weights = np.exp(scores_shifted) / np.sum(np.exp(scores_shifted), axis=-1, keepdims=True)
        
        # Apply attention to values
        head_output = np.dot(attention_weights, V[:, h, :])
        heads.append(head_output)
    
    # Concatenate heads
    concat_heads = np.concatenate(heads, axis=-1)
    
    # Final projection
    output = np.dot(concat_heads, W_O)
    
    return output

# Example usage
N, d = 4, 8
num_heads = 2
d_k = d // num_heads

X = np.random.randn(N, d)
W_Q = np.random.randn(d, num_heads * d_k)
W_K = np.random.randn(d, num_heads * d_k)
W_V = np.random.randn(d, num_heads * d_k)
W_O = np.random.randn(num_heads * d_k, d)

output = multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads, d_k)
print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")

Input shape: (4, 8)
Output shape: (4, 8)


<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# Part 4: Input and Output - Position Embeddings and the Language Model Head

Now let's understand how inputs are encoded and outputs are generated in transformers.

# Token and Position Embeddings

The matrix $\mathbf{X}$ (of shape $[N \times d]$) has an embedding for each word in the context.

This embedding is created by **adding two distinct embeddings** for each input:
- **Token embedding**
- **Positional embedding**

# Token Embeddings

Embedding matrix $\mathbf{E}$ has shape $[|V| \times d]$:
- One row for each of the $|V|$ tokens in the vocabulary
- Each word is a row vector of $d$ dimensions

**Example**: Given string "Thanks for all the"

1. Tokenize with BPE and convert into vocab indices:  
   $w = [5, 4000, 10532, 2224]$

2. Select the corresponding rows from $\mathbf{E}$, each row an embedding:
   - (row 5, row 4000, row 10532, row 2224)

# Position Embeddings

There are many methods, but we'll describe the simplest: **absolute position**.

**Goal**: Learn a position embedding matrix $\mathbf{E}_{pos}$ of shape $[1 \times N]$.

Start with randomly initialized embeddings:
- One for each integer up to some maximum length
- i.e., just as we have an embedding for token *fish*, we'll have an embedding for position 3 and position 17
- As with word embeddings, these position embeddings are learned along with other parameters during training

> **Modern alternative — RoPE (Rotary Position Embeddings):**
> Most current LLMs (Llama, Mistral, Qwen, Gemma) use **RoPE** (Su et al., 2021) instead of absolute position embeddings. RoPE encodes position by *rotating* query and key vectors in 2D subspaces by angles proportional to their position. The dot product between q at position m and k at position n naturally depends only on the *relative distance* (m − n). Key advantage: models trained with short contexts can be extended to much longer contexts at inference using techniques like **YaRN** or **NTK-aware scaling**.


<center>
<img width=80% src="img/word-position-embedding.png">
</center>

# Byte Pair Encoding (BPE)

**Goal.** Build a subword vocabulary so any text can be tokenized (few OOVs) with a manageable vocab size.

**Training (learn merges once):**
1. Start with characters (plus an end-of-word marker like `</w>`).
2. Count all adjacent symbol pairs in a large corpus.
3. Merge the most frequent pair into a new symbol; add it to the vocab.
4. Repeat steps 2–3 for \(M\) merges → final vocab size \(|V| = |\text{chars}| + M\).

**Tokenization at inference (apply learned merges):**
For each word, repeatedly apply the learned merges (most-specific first) until no merge matches.

---

## Example: “Thanks for all the”

*(Toy illustration of merges; exact steps depend on the learned merge list.)*

- `Thanks</w>`  
  `T h a n k s </w>` → `Th a n k s </w>` → `Tha nk s </w>` → `Than ks </w>` → `Thanks </w>` → `Thanks`
- `for</w>`  
  `f o r </w>` → `fo r </w>` → `for </w>` → `for`
- `all</w>`  
  `a l l </w>` → `al l </w>` → `all`
- `the</w>`  
  `t h e </w>` → `the`

**Resulting tokens:** `["Thanks", "for", "all", "the"]`

**Map to vocab ids (from previous slide):**  
\(w = [5,\; 4000,\; 10532,\; 2224]\)

---

### Notes
- BPE mixes whole words and subwords (e.g., `unbelievable` → `un`, `believ`, `able`).
- Spaces are handled via markers (e.g., `</w>` or a leading-space token like `Ġ` in GPT-style vocabularies).
- After tokenization, look up embeddings by id in \(\mathbf{E}\) just as shown on the prior slide.


In [20]:
# Position embedding demonstration
def create_positional_embeddings(max_len, d_model):
    """
    Create learnable positional embeddings.
    
    Args:
        max_len: Maximum sequence length
        d_model: Model dimension
    
    Returns:
        Position embedding matrix
    """
    # Initialize randomly (in practice, these are learned)
    pos_embeddings = np.random.randn(max_len, d_model) * 0.02
    return pos_embeddings

# Example: Combine token and position embeddings
vocab_size = 50000
d_model = 512
max_len = 1024

# Token embedding matrix
token_embeddings = np.random.randn(vocab_size, d_model) * 0.02

# Position embedding matrix
pos_embeddings = create_positional_embeddings(max_len, d_model)

# Example sentence: [5, 4000, 10532, 2224]
tokens = [5, 4000, 10532, 2224]
seq_len = len(tokens)

# Get token embeddings
token_emb = token_embeddings[tokens]

# Get position embeddings
position_emb = pos_embeddings[:seq_len]

# Combine
input_embeddings = token_emb + position_emb

print(f"Token embeddings shape: {token_emb.shape}")
print(f"Position embeddings shape: {position_emb.shape}")
print(f"Combined input embeddings shape: {input_embeddings.shape}")

Token embeddings shape: (4, 512)
Position embeddings shape: (4, 512)
Combined input embeddings shape: (4, 512)


# Language modeling head: Unembedding layer

**Unembedding layer**: Linear layer projects from $\mathbf{h}_N^L$ (shape $[1 \times d]$) to logit vector

**Why "unembedding"?** Tied to $\mathbf{E}^T$

**Weight tying**: We use the same weights for two different matrices

Unembedding layer maps from an embedding to a $1 \times |V|$ vector of logits

- Embedding matrix $\mathbf{E}$ (shape $[|V| \times d]$): maps from vocabulary to embedding
- Unembedding matrix $\mathbf{E}^T$ (shape $[d \times |V|]$): maps from embedding back to vocabulary


<center>
<img width=80% src="img/lm-unembedding.png">
</center>

<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c"
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# Part 5: Modern Developments (2022–2026)

The transformer architecture has evolved significantly since the original "Attention Is All You Need" paper. This section covers the most important developments that define how transformers are built and deployed today.

# Flash Attention: Making Attention Fast

**Problem**: Standard attention is **memory-bandwidth bound**, not compute-bound. Writing/reading the N×N attention matrix to/from GPU memory is the bottleneck.

**Solution** (Dao et al., 2022): **Flash Attention** restructures the computation into tiles that fit in GPU SRAM (on-chip memory), avoiding materializing the full attention matrix in HBM (main GPU memory).

**Key properties:**
- Computes the **exact same result** as standard attention — no approximation
- 2-4x wall-clock speedup; enables much longer context lengths
- Flash Attention 2 (2023): better parallelism and work partitioning
- Flash Attention 3 (2024): exploits Hopper GPU features (async execution, FP8)

**Takeaway**: This is a *systems-level* optimization, not a mathematical change to attention. Understanding the memory hierarchy of modern GPUs is essential for efficient deep learning.

# Grouped-Query Attention (GQA) and Multi-Query Attention (MQA)

**Problem**: During autoregressive generation, the **KV cache** stores key-value pairs for all past tokens. With many attention heads, this cache becomes a memory bottleneck.

| Variant | KV Heads | Used In |
|---------|----------|---------|
| **Multi-Head Attention (MHA)** | Same as query heads (e.g., 32) | Original Transformer |
| **Multi-Query Attention (MQA)** | 1 shared KV head | PaLM, StarCoder |
| **Grouped-Query Attention (GQA)** | Groups of query heads share KV (e.g., 8 KV for 32 Q) | Llama 2/3, Mistral, Gemma |

**GQA** (Ainslie et al., 2023) is the sweet spot: it dramatically reduces KV cache memory while preserving nearly all of MHA's quality. It has become the **default choice** for production LLMs.

**Why it matters for inference:**
- Smaller KV cache → longer context windows fit in memory
- Smaller KV cache → higher throughput (more requests per GPU)

# The KV Cache: Why Inference Is Different from Training

During **training**, we process all tokens in parallel (using the causal mask).

During **inference** (autoregressive generation), we generate one token at a time:
1. For each new token, compute its Q, K, V
2. Compute attention against **all previous** K and V vectors
3. Rather than recomputing all previous K/V, we **cache** them — the **KV cache**

**KV cache size** = $2 \times n_{\text{layers}} \times n_{\text{kv\_heads}} \times d_{\text{head}} \times n_{\text{tokens}} \times \text{precision\_bytes}$

For Llama 3 70B with 128K context: KV cache alone can require **~40 GB**!

**Optimizations:**
- **GQA/MQA**: Fewer KV heads → smaller cache
- **Paged Attention** (vLLM): Manages KV cache like virtual memory pages — eliminates fragmentation
- **KV cache quantization**: Store cached values in INT8/INT4
- **Sliding window**: Only cache the last W tokens per layer

# Mixture of Experts (MoE)

**Idea**: Replace each dense FFN layer with N "expert" FFN sub-networks + a lightweight **router** that selects the top-k experts per token.

**Result**: The model has many total parameters but only activates a fraction per token.

| Model | Total Params | Active Params/Token | Experts (active) |
|-------|-------------|--------------------|----|
| Mixtral 8x7B | 46.7B | ~13B | 8 (2) |
| DeepSeek-V3 | 671B | ~37B | 256 (8) |

**Why MoE matters:**
- Decouples **capacity** (total params = knowledge stored) from **compute cost** (active params per token)
- Mixtral 8x7B matches Llama 2 70B quality while being much faster
- MoE is now the **dominant architecture** for frontier models

**Recent innovations** (2024-2025):
- Fine-grained experts (more, smaller experts)
- Shared/always-on expert layers
- Auxiliary-loss-free load balancing (DeepSeek-V3)

<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c"
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# Optional Reading: Modern Developments (2022–2026)

The transformer architecture has evolved significantly since the original "Attention Is All You Need" paper. This section covers the most important developments that define how transformers are built and deployed today.

# Speculative Decoding: Faster Inference Without Quality Loss

**Problem**: Autoregressive generation is slow because each token requires a full forward pass through the model, but this underutilizes GPU compute (it's memory-bandwidth bound).

**Solution**: Use a small, fast **draft model** to generate k tokens ahead, then have the large **target model** verify all k tokens in a **single parallel forward pass**.

**Algorithm:**
1. Draft model generates k candidate tokens: $t_1, t_2, ..., t_k$
2. Target model scores all k tokens in one forward pass
3. Accept tokens from left until first disagreement
4. On rejection, sample from the target model's corrected distribution

**Key property**: The output distribution is **mathematically identical** to running the target model alone — this is **lossless** acceleration!

**Speedup**: Typically 2-3x, depending on draft model agreement rate.

**Variants**: Self-speculative decoding (early exit), Medusa (multiple heads), EAGLE (feature-level drafting)

# Beyond Attention: State Space Models (Mamba)

**Key question**: Is the quadratic attention mechanism truly necessary?

**State Space Models** (SSMs) model sequences through continuous-time state equations with **O(N) complexity** and **constant memory** at inference (no KV cache).

**Mamba** (Gu & Dao, 2023): Made SSMs competitive with transformers by introducing **selective** state spaces where parameters are input-dependent (analogous to attention's content-based routing).

**Strengths of SSMs:**
- Linear time complexity for long sequences
- No KV cache needed — constant memory at inference
- Excellent for streaming / real-time applications

**Limitations:**
- Struggle with precise **retrieval** from context ("needle in a haystack")
- Pure SSMs haven't matched transformers at large scale

**The convergence (2024-2025):** Hybrid architectures combining SSM + attention layers:
- **Jamba** (AI21): Interleaves Mamba, attention, and MoE layers
- **Mamba-2**: Formalized the mathematical connection between SSMs and structured attention

# Test-Time Compute Scaling: A New Scaling Paradigm

Traditionally, we scale LLMs by increasing **model size** and **training data**.

**New paradigm** (2024-2025): Scale **inference-time compute** instead.

**How it works:**
- Models generate longer **chains of reasoning** before answering
- Search over multiple solution paths (tree of thought)
- Self-verification and iterative refinement

**Examples:** OpenAI o1/o3, DeepSeek-R1, Claude extended thinking

**Key finding** (Snell et al., 2024): For reasoning-heavy tasks, spending more compute at inference can be **more efficient** than increasing model size.

**DeepSeek-R1** (2025): Demonstrated that chain-of-thought reasoning can emerge from **pure reinforcement learning** without supervised reasoning traces.

**Implication**: The future of LLM improvement may rely as much on **inference strategies** as on model architecture and training.

# Summary: The Modern Transformer Stack (2026)

A state-of-the-art transformer LLM today typically includes:

| Component | Original (2017) | Modern (2024-2026) |
|-----------|----------------|-------------------|
| **Attention** | Multi-head (MHA) | Grouped-Query (GQA) + Flash Attention |
| **Position encoding** | Sinusoidal / Learned absolute | RoPE (Rotary) |
| **Normalization** | Post-LayerNorm | Pre-RMSNorm |
| **FFN activation** | ReLU | SwiGLU |
| **Architecture** | Dense | Mixture of Experts (MoE) |
| **Training precision** | FP32 → FP16 | BF16 → FP8 |
| **Inference** | Naive autoregressive | KV cache + speculative decoding + quantization |
| **Scaling strategy** | Bigger models | Inference-time compute + overtraining smaller models |

The core self-attention mechanism remains, but almost every other component has been improved or replaced.

In [ ]:
# Comparison: MHA vs GQA vs MQA — KV cache sizes
import numpy as np
import matplotlib.pyplot as plt

def kv_cache_size_gb(n_layers, n_kv_heads, d_head, seq_len, precision_bytes=2):
    """Calculate KV cache size in GB."""
    # 2 for K and V
    return 2 * n_layers * n_kv_heads * d_head * seq_len * precision_bytes / (1024**3)

# Llama 3 70B-like config
n_layers = 80
n_query_heads = 64
d_head = 128
seq_lengths = [4096, 8192, 16384, 32768, 65536, 131072]

configs = {
    "MHA (64 KV heads)": 64,
    "GQA (8 KV heads)": 8,
    "MQA (1 KV head)": 1,
}

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

for label, n_kv in configs.items():
    sizes = [kv_cache_size_gb(n_layers, n_kv, d_head, sl) for sl in seq_lengths]
    ax.plot([s // 1024 for s in seq_lengths], sizes, 'o-', label=label, linewidth=2)

ax.set_xlabel("Context Length (K tokens)", fontsize=12)
ax.set_ylabel("KV Cache Size (GB)", fontsize=12)
ax.set_title("KV Cache Memory: MHA vs GQA vs MQA\n(Llama 3 70B-scale model, BF16)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks([s // 1024 for s in seq_lengths])
plt.tight_layout()
plt.show()

print("At 128K context length:")
for label, n_kv in configs.items():
    size = kv_cache_size_gb(n_layers, n_kv, d_head, 131072)
    print(f"  {label}: {size:.1f} GB")

# References and Credits

This lecture is based on materials from:

**Primary Source**:
Daniel Jurafsky and James H. Martin. 2025. *Speech and Language Processing: An Introduction to Natural Language Processing, Computational Linguistics, and Speech Recognition with Language Models*, 3rd edition. Online manuscript released August 24, 2025. https://web.stanford.edu/~jurafsky/slp3.

**Original Transformer Paper**:
Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). *Attention is all you need*. In Advances in Neural Information Processing Systems (pp. 5998-6008).

**Additional References**:
- Elhage, N., et al. (2021). *A Mathematical Framework for Transformer Circuits*. Anthropic.
- Kaplan, J., et al. (2020). *Scaling Laws for Neural Language Models*. arXiv:2001.08361.
- Hu, E. J., et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*. arXiv:2106.09685.

**Modern Developments (added Spring 2026)**:
- Dao, T. (2023). *FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning*.
- Ainslie, J., et al. (2023). *GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints*. EMNLP 2023.
- Su, J., et al. (2023). *RoFormer: Enhanced Transformer with Rotary Position Embedding*.
- Jiang, A., et al. (2024). *Mixtral of Experts*.
- Gu, A. & Dao, T. (2023). *Mamba: Linear-Time Sequence Modeling with Selective State Spaces*.
- Kwon, W., et al. (2023). *Efficient Memory Management for Large Language Model Serving with PagedAttention*. SOSP 2023.
- Leviathan, Y., et al. (2023). *Fast Inference from Transformers via Speculative Decoding*. ICML 2023.
- Snell, C., et al. (2024). *Scaling LLM Test-Time Compute Optimally Can Be More Effective Than Scaling Model Parameters*.
- DeepSeek-AI. (2024). *DeepSeek-V3 Technical Report*.
- Shazeer, N. (2020). *GLU Variants Improve Transformer*.